# 在 Intel Xeon CPU 上使用 Langchain 和开源模型本地运行 RAG 应用

This guide will walk you through setting up a Retrieval-Augmented Generation (RAG) application locally on your Intel Xeon CPU. We'll be using Langchain, a popular framework for developing applications powered by language models, and leveraging open-source models for text generation and embedding.

本指南将引导您在 Intel Xeon CPU 上本地设置一个检索增强生成 (RAG) 应用。我们将使用 Langchain，一个用于开发语言模型驱动型应用的流行框架，并利用开源模型进行文本生成和嵌入。

**Prerequisites**

**先决条件**

*   **Python 3.8+**: Ensure you have a recent version of Python installed.
*   **Python 3.8+**: 确保您已安装最新版本的 Python。
*   **pip**: Python's package installer.
*   **pip**: Python 的包安装程序。
*   **Git**: For cloning repositories.
*   **Git**: 用于克隆仓库。
*   **Basic understanding of the command line**: Familiarity with terminal commands.
*   **对命令行有基本了解**: 熟悉终端命令。

**1. Setting up the Environment**

**1. 设置环境**

First, let's create a virtual environment to keep our project dependencies isolated.

首先，让我们创建一个虚拟环境来隔离我们的项目依赖。

```bash
python -m venv venv
source venv/bin/activate  # On Windows use `venv\Scripts\activate`
```

```bash
python -m venv venv
source venv/bin/activate  # 在 Windows 上使用 `venv\Scripts\activate`
```

Next, install the necessary Python packages.

接下来，安装必要的 Python 包。

```bash
pip install langchain openai chromadb sentence-transformers pypdf unstructured
```

```bash
pip install langchain openai chromadb sentence-transformers pypdf unstructured
```

*   `langchain`: The core framework.
*   `langchain`: 核心框架。
*   `openai`: Although we're using open-source models, Langchain's `OpenAI` class is often used as a base or for comparison. We'll configure it to use a local model later.
*   `openai`: 尽管我们正在使用开源模型，但 Langchain 的 `OpenAI` 类经常被用作基础或用于比较。我们稍后会将其配置为使用本地模型。
*   `chromadb`: An open-source embedding database for storing and retrieving document chunks.
*   `chromadb`: 一个开源嵌入数据库，用于存储和检索文档片段。
*   `sentence-transformers`: A library for generating sentence and text embeddings using pre-trained models.
*   `sentence-transformers`: 一个使用预训练模型生成句子和文本嵌入的库。
*   `pypdf`: For reading PDF files.
*   `pypdf`: 用于读取 PDF 文件。
*   `unstructured`: For parsing various document types.
*   `unstructured`: 用于解析各种文档类型。

**2. Downloading an Open-Source LLM and Embedding Model**

**2. 下载开源 LLM 和嵌入模型**

For local execution, we'll use models that can run efficiently on CPU. `sentence-transformers` provides access to many pre-trained embedding models. For the LLM, we can use models compatible with libraries like `ctransformers` or `llama-cpp-python`.

为了本地运行，我们将使用可以在 CPU 上高效运行的模型。`sentence-transformers` 提供了许多预训练嵌入模型的访问权限。对于 LLM，我们可以使用与 `ctransformers` 或 `llama-cpp-python` 等库兼容的模型。

Let's choose a sentence transformer model for embeddings and a small, efficient LLM for generation.

让我们选择一个用于嵌入的句子转换器模型和一个用于生成的、小型高效的 LLM。

**Embedding Model:** `all-MiniLM-L6-v2` is a good starting point.

**嵌入模型**: `all-MiniLM-L6-v2` 是一个不错的起点。

**LLM:** We'll use a quantized version of a model like Llama 2 or Mistral. For this example, let's assume you've downloaded a GGUF-formatted model (e.g., `mistral-7b-instruct-v0.1.Q4_K_M.gguf`) from Hugging Face. You'll need to place this file in a known directory.

**LLM**: 我们将使用像 Llama 2 或 Mistral 这样的模型的量化版本。在本例中，我们假设您已从 Hugging Face 下载了 GGUF 格式的模型（例如 `mistral-7b-instruct-v0.1.Q4_K_M.gguf`）。您需要将此文件放在一个已知目录中。

**3. Preparing Your Documents**

**3. 准备您的文档**

Create a directory named `data` and place your documents (e.g., PDFs, text files) inside it.

创建一个名为 `data` 的目录，并将您的文档（例如 PDF、文本文件）放在其中。

**4. Building the RAG Application**

**4. 构建 RAG 应用**

Now, let's create a Python script (e.g., `rag_app.py`) to implement the RAG pipeline.

现在，让我们创建一个 Python 脚本（例如 `rag_app.py`）来实现 RAG 管道。

```python
import os
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import LlamaCpp
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# --- Configuration ---
DATA_PATH = "data"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
# Replace with the actual path to your downloaded GGUF model
LLM_MODEL_PATH = "/path/to/your/mistral-7b-instruct-v0.1.Q4_K_M.gguf"
CHROMA_DB_PATH = "chroma_db"

# --- 1. Load Documents ---
def load_documents(data_path):
    loader = DirectoryLoader(data_path, glob="**/*.*", show_progress=True, use_multithreading=True)
    documents = loader.load()
    print(f"Loaded {len(documents)} documents.")
    return documents

# --- 2. Split Documents ---
def split_documents(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    texts = text_splitter.split_documents(documents)
    print(f"Split into {len(texts)} chunks.")
    return texts

# --- 3. Create Embeddings ---
def create_embeddings(texts):
    # Use HuggingFaceEmbeddings for local embedding generation
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
    print(f"Using embedding model: {EMBEDDING_MODEL_NAME}")
    return embeddings

# --- 4. Create Vector Store ---
def create_vector_store(texts, embeddings, db_path):
    if not os.path.exists(db_path):
        print(f"Creating vector store at {db_path}...")
        vector_store = Chroma.from_documents(
            documents=texts,
            embedding=embeddings,
            persist_directory=db_path
        )
        vector_store.persist()
        print("Vector store created and persisted.")
    else:
        print(f"Loading existing vector store from {db_path}...")
        vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)
        print("Vector store loaded.")
    return vector_store

# --- 5. Initialize LLM ---
def initialize_llm(model_path):
    print(f"Initializing LLM from: {model_path}")
    # Configure LlamaCpp for local CPU inference
    llm = LlamaCpp(
        model_path=model_path,
        n_gpu_layers=0,  # Set to 0 for CPU inference
        n_batch=512,     # Batch size for prompt processing
        n_ctx=2048,      # Context window size
        f16_kv=False,    # Set to False for CPU
        verbose=True,    # Enable verbose logging
    )
    print("LLM initialized.")
    return llm

# --- 6. Create QA Chain ---
def create_qa_chain(llm, vector_store):
    retriever = vector_store.as_retriever()

    # Define a prompt template
    template = """
    Use the following pieces of context to answer the question at the end.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.
    Be concise and stick to the context provided.

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
    QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

    qa_chain = RetrievalQA.from_chain_type(
        llm,
        retriever=retriever,
        chain_type_kwargs={"prompt": QA_CHAIN_PROMPT},
        return_source_documents=True # Optional: to see which documents were used
    )
    print("QA chain created.")
    return qa_chain

# --- Main Execution ---
if __name__ == "__main__":
    # Ensure the data directory exists
    if not os.path.exists(DATA_PATH):
        print(f"Error: Data directory '{DATA_PATH}' not found. Please create it and add your documents.")
        exit()

    # Ensure the LLM model file exists
    if not os.path.exists(LLM_MODEL_PATH):
        print(f"Error: LLM model file not found at '{LLM_MODEL_PATH}'. Please download it and update the path.")
        exit()

    # 1. Load Documents
    documents = load_documents(DATA_PATH)

    # 2. Split Documents
    texts = split_documents(documents)

    # 3. Create Embeddings
    embeddings = create_embeddings(texts)

    # 4. Create or Load Vector Store
    vector_store = create_vector_store(texts, embeddings, CHROMA_DB_PATH)

    # 5. Initialize LLM
    llm = initialize_llm(LLM_MODEL_PATH)

    # 6. Create QA Chain
    qa_chain = create_qa_chain(llm, vector_store)

    # --- Interact with the RAG App ---
    print("\n--- RAG Application Ready ---")
    print("Ask a question based on your documents. Type 'exit' to quit.")

    while True:
        query = input("\nQuestion: ")
        if query.lower() == 'exit':
            break

        # Run the QA chain
        result = qa_chain({"query": query})

        print("\nAnswer:")
        print(result["result"])

        # Optional: Print source documents
        # print("\nSource Documents:")
        # for doc in result["source_documents"]:
        #     print(f"- {doc.metadata.get('source', 'N/A')}: {doc.page_content[:100]}...") # Print first 100 chars

```

**5. Running the Application**

**5. 运行应用**

1.  **Place your documents**: Put your text files or PDFs into the `data` directory.
2.  **Place your LLM**: Ensure your downloaded GGUF model file is in the location specified by `LLM_MODEL_PATH` in the script.
3.  **Run the script**:
    ```bash
    python rag_app.py
    ```

The script will first process your documents, create embeddings, and build/load the ChromaDB. This might take a few minutes depending on the number and size of your documents. Once ready, it will prompt you to ask questions.

**Example Interaction:**

**示例交互：**

```
Loaded 5 documents.
Split into 25 chunks.
Using embedding model: sentence-transformers/all-MiniLM-L6-v2
Creating vector store at chroma_db...
Vector store created and persisted.
Initializing LLM from: /path/to/your/mistral-7b-instruct-v0.1.Q4_K_M.gguf
llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from /path/to/your/mistral-7b-instruct-v0.1.Q4_K_M.gguf (version GGUF of ...):
llama_model_loader:                            internal_id: UNKNOWN
llama_model_loader:                            context_params: n_ctx=2048, n_embd=4096, n_head=32, n_layer=32, n_vocab=32000
llama_model_loader:                            ggml ctx size = 0.736 MB
llama_model_loader:                            Using CPU for computation
llama_model_loader:                            Vocab size: 32000
llama_model_loader:                            Offset of token to id mapping: 0
llama_model_loader:                            Offset of special tokens mapping: 32000
llama_model_loader:                            Loading model weights from...
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice: MADVISE
llama_model_loader:                            Using mmap advice:

作者 - Pratool Bharti (pratool.bharti@intel.com)

在本食谱中，我们使用 langchain 工具和开源模型在 CPU 上本地执行。本笔记本已在 Intel Xeon 8480+ CPU 上验证通过。在此，我们为 Llama2 模型实现了一个 RAG 流水线，以回答有关 Intel 2024 年第一季度财报发布的问题。

**创建 conda 或 virtualenv 环境，并安装以下库（Python 版本 >= 3.10）**
<br>

`pip install --upgrade langchain langchain-community langchainhub langchain-chroma bs4 gpt4all pypdf pysqlite3-binary` <br>
`pip install llama-cpp-python   --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu`

**将 pysqlite3 加载到 sys 模块中，因为 ChromaDB 需要 sqlite3。**

In [1]:
__import__("pysqlite3")
import sys

sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

**导入必要的组件以加载和拆分数据**

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

**下载英特尔 2024 年第一季度财报**

In [4]:
!wget  'https://d1io3yog0oux5.cloudfront.net/_11d435a500963f99155ee058df09f574/intel/db/887/9014/earnings_release/Q1+24_EarningsRelease_FINAL.pdf' -O intel_q1_2024_earnings.pdf

--2024-07-15 15:04:43--  https://d1io3yog0oux5.cloudfront.net/_11d435a500963f99155ee058df09f574/intel/db/887/9014/earnings_release/Q1+24_EarningsRelease_FINAL.pdf
Resolving proxy-dmz.intel.com (proxy-dmz.intel.com)... 10.7.211.16
Connecting to proxy-dmz.intel.com (proxy-dmz.intel.com)|10.7.211.16|:912... connected.
Proxy request sent, awaiting response... 200 OK
Length: 133510 (130K) [application/pdf]
Saving to: ‘intel_q1_2024_earnings.pdf’

intel_q1_2024_earni 100%[===================>] 130.38K  --.-KB/s    in 0.005s  

2024-07-15 15:04:44 (24.6 MB/s) - ‘intel_q1_2024_earnings.pdf’ saved [133510/133510]



**使用 PyPDFLoader 加载收益发布 PDF 文档**

In [5]:
loader = PyPDFLoader("intel_q1_2024_earnings.pdf")
data = loader.load()

**将整个文档拆分成几个块，每个块的大小为 500 个 token**

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
all_splits = text_splitter.split_documents(data)

**查看文档的第一个拆分**

In [7]:
all_splits[0]

Document(metadata={'source': 'intel_q1_2024_earnings.pdf', 'page': 0}, page_content='Intel Corporation\n2200 Mission College Blvd.\nSanta Clara, CA 95054-1549\n                                                         \nNews Release\n Intel Reports First -Quarter 2024  Financial Results\nNEWS SUMMARY\n▪First-quarter revenue of $12.7 billion , up 9%  year over year (YoY).\n▪First-quarter GAAP earnings (loss) per share (EPS) attributable to Intel was $(0.09) ; non-GAAP EPS \nattributable to Intel was $0.18 .')

**RAG 的一个主要步骤是将文档的每个分块转换为嵌入，并将其存储在向量数据库中，以便高效地搜索相关文档。** <br>
**为此，从 langchain 导入 Chroma 向量数据库。同时，导入开源的 GPT4All 用于嵌入模型。**

In [8]:
from langchain_chroma import Chroma
from langchain_community.embeddings import GPT4AllEmbeddings

接下来，我们将下载一个最受欢迎的嵌入模型“all-MiniLM-L6-v2”。您可以在此链接找到该模型的更多详细信息：https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

In [10]:
model_name = "all-MiniLM-L6-v2.gguf2.f16.gguf"
gpt4all_kwargs = {"allow_download": "True"}
embeddings = GPT4AllEmbeddings(model_name=model_name, gpt4all_kwargs=gpt4all_kwargs)

**将所有嵌入存储在 Chroma 数据库中**

In [11]:
vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings)

**现在，让我们从文档中查找与问题相关的拆分**

In [12]:
question = "What is Intel CCG revenue in Q1 2024"
docs = vectorstore.similarity_search(question)
print(len(docs))

4


**查看从向量数据库检索到的第一个文档**

In [13]:
docs[0]

Document(metadata={'page': 1, 'source': 'intel_q1_2024_earnings.pdf'}, page_content='Client Computing Group (CCG) $7.5 billion up31%\nData Center and AI (DCAI) $3.0 billion up5%\nNetwork and Edge (NEX) $1.4 billion down 8%\nTotal Intel Products revenue $11.9 billion up17%\nIntel Foundry $4.4 billion down 10%\nAll other:\nAltera $342 million down 58%\nMobileye $239 million down 48%\nOther $194 million up17%\nTotal all other revenue $775 million down 46%\nIntersegment eliminations $(4.4) billion\nTotal net revenue $12.7 billion up9%\nIntel Products Highlights')

**从 Huggingface 下载 Llama-2 模型并本地存储** <br>
**您可以从下面的链接下载 Llama-2 模型的不同量化版本。我们这里使用的是 Q8 版本（7.16GB）。** <br>
https://huggingface.co/TheBloke/Llama-2-7B-Chat-GGUF

In [ ]:
!huggingface-cli download TheBloke/Llama-2-7b-Chat-GGUF llama-2-7b-chat.Q8_0.gguf --local-dir . --local-dir-use-symlinks False

**导入加载已下载的 LLM 模型所需的 langchain 组件**

In [14]:
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

**使用 Llama-cpp 库加载本地 Llama-2 模型**

In [16]:
llm = LlamaCpp(
    model_path="llama-2-7b-chat.Q8_0.gguf",
    n_gpu_layers=-1,
    n_batch=512,
    n_ctx=2048,
    f16_kv=True,  # MUST set to True, otherwise you will run into problem after a couple of calls
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from llama-2-7b-chat.Q8_0.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u32              = 32

**现在，我们向 Llama 模型提出同样的问题，但不向它们展示收益报告。**

In [17]:
llm.invoke(question)

?
(NASDAQ:INTC)
Intel's CCG (Client Computing Group) revenue for Q1 2024 was $9.6 billion, a decrease of 35% from the previous quarter and a decrease of 42% from the same period last year.


llama_print_timings:        load time =     131.20 ms
llama_print_timings:      sample time =      16.05 ms /    68 runs   (    0.24 ms per token,  4236.76 tokens per second)
llama_print_timings: prompt eval time =     131.14 ms /    16 tokens (    8.20 ms per token,   122.01 tokens per second)
llama_print_timings:        eval time =    3225.00 ms /    67 runs   (   48.13 ms per token,    20.78 tokens per second)
llama_print_timings:       total time =    3466.40 ms /    83 tokens


"?\n(NASDAQ:INTC)\nIntel's CCG (Client Computing Group) revenue for Q1 2024 was $9.6 billion, a decrease of 35% from the previous quarter and a decrease of 42% from the same period last year."

正如您所见，模型给出了错误的信息。正确答案是 CCG 2024 年第一季度的收入为 75 亿美元。现在，让我们使用收益发布文件应用 RAG。

在 RAG 中，我们通过添加相关文档来修改输入提示，并附带问题。这里，我们使用了一个流行的 RAG 提示。

In [18]:
from langchain import hub

rag_prompt = hub.pull("rlm/rag-prompt")
rag_prompt.messages

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"))]

**将所有检索到的文档合并到一个文档中**

In [19]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

最后一步是使用langchain工具创建一个链，该链将创建一个端到端的管道。它将以问题和上下文作为输入。

In [20]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnablePick

# Chain
chain = (
    RunnablePassthrough.assign(context=RunnablePick("context") | format_docs)
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [21]:
chain.invoke({"context": docs, "question": question})

Llama.generate: prefix-match hit


 Based on the provided context, Intel CCG revenue in Q1 2024 was $7.5 billion up 31%.


llama_print_timings:        load time =     131.20 ms
llama_print_timings:      sample time =       7.74 ms /    31 runs   (    0.25 ms per token,  4004.13 tokens per second)
llama_print_timings: prompt eval time =    2529.41 ms /   674 tokens (    3.75 ms per token,   266.46 tokens per second)
llama_print_timings:        eval time =    1542.94 ms /    30 runs   (   51.43 ms per token,    19.44 tokens per second)
llama_print_timings:       total time =    4123.68 ms /   704 tokens


' Based on the provided context, Intel CCG revenue in Q1 2024 was $7.5 billion up 31%.'

**现在我们看到结果是正确的，正如收益报告中所述。** <br>
**为了进一步自动化，我们将创建一个链，该链将以问题和检索器作为输入，这样我们就无需单独检索文档。**

In [22]:
retriever = vectorstore.as_retriever()
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

**现在我们只需要将问题传递给链，它将直接从向量数据库中获取上下文来生成答案**
<br>
**让我们尝试另一个问题**

In [26]:
qa_chain.invoke("what is Intel DCAI revenue in Q1 2024?")

Llama.generate: prefix-match hit


 According to the provided context, Intel DCAI revenue in Q1 2024 was $3.0 billion up 5%.


llama_print_timings:        load time =     131.20 ms
llama_print_timings:      sample time =       6.28 ms /    31 runs   (    0.20 ms per token,  4937.88 tokens per second)
llama_print_timings: prompt eval time =    2681.93 ms /   730 tokens (    3.67 ms per token,   272.19 tokens per second)
llama_print_timings:        eval time =    1471.07 ms /    30 runs   (   49.04 ms per token,    20.39 tokens per second)
llama_print_timings:       total time =    4206.77 ms /   760 tokens


' According to the provided context, Intel DCAI revenue in Q1 2024 was $3.0 billion up 5%.'